# Redrob Hackathon — Kaggle Dual-GPU Pipeline

**Goal:** Rank top 100 from 100K for Senior AI Engineer.

**Kaggle setup:** *Settings → Accelerator → GPU T4 x2*

### Pipeline
1. Clone + extract dataset
2. Install deps
3. GPU precompute: bi-encoder (DataParallel) + BM25 + cross-encoder + features
4. CPU rank: XGBoost LTR → top 100
5. Validate + download

## Step 1 — Clone + Extract

In [ ]:
!git clone https://github.com/JASHWANTHS07/india-runs.git
%cd india-runs
!apt-get install -qq unrar
!unrar x -o+ dataset/Data.rar dataset/
!ls dataset/India_runs_data_and_ai_challenge/

In [ ]:
import os
CANDIDATES = "dataset/India_runs_data_and_ai_challenge/candidates.jsonl"
size_mb = os.path.getsize(CANDIDATES) / 1e6
print(f"candidates.jsonl: {size_mb:.1f} MB")
assert size_mb > 400
print("Dataset OK")

## Step 2 — Install Dependencies + GPU Check

In [ ]:
!pip install -q sentence-transformers tqdm pandas pyarrow xgboost optuna

import torch
gpu_count = torch.cuda.device_count()
for i in range(gpu_count):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_mem / 1e9
    print(f"  GPU {i}: {name} ({mem:.1f} GB)")
print(f"Total GPUs: {gpu_count}")

## Step 3 — Pre-compute (Dual GPU)

Bi-encoder embeddings with DataParallel + BM25 + cross-encoder re-ranking + 78 LTR features + 3 reasoning fields (notable_company, best_institution, best_field).

Features: education, certs, career trajectory, skill breadth, work mode, behavioral signals, notable company history, headline analysis, salary fit, assessment depth, market demand, summary template detection, career description coherence.

**Expected:** ~15–25 min on 2x T4.

In [1]:
import json, sys, time, os
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder

In [ ]:
# Ensure we're in the repo directory
if not os.path.exists('src/features.py'):
    os.chdir('india-runs')
print(f'Working dir: {os.getcwd()}')

CANDIDATES = "dataset/India_runs_data_and_ai_challenge/candidates.jsonl"
ARTIFACTS = "./artifacts"
os.makedirs(ARTIFACTS, exist_ok=True)
BATCH_SIZE = 256
CROSS_ENCODER_TOP_N = 1000

# ---- Load candidates & extract features ----
print("Loading candidates + extracting features...")
sys.path.insert(0, '.')
from src.features import extract_features
from src.bm25 import compute_bm25_scores
from src.precompute import JD_QUERY
from dataclasses import asdict

candidates = []
with open(CANDIDATES) as f:
    for line in f:
        candidates.append(json.loads(line))

feat_dicts = []
all_texts = []
for cand in tqdm(candidates, desc="Features"):
    fd = asdict(extract_features(cand))
    feat_dicts.append(fd)
    all_texts.append(fd.get("profile_text", ""))
print(f"  {len(candidates)} candidates loaded")

# ---- BM25 ----
print("Computing BM25 scores...")
t_bm25 = time.time()
bm25_scores = compute_bm25_scores(all_texts, JD_QUERY)
print(f"  BM25 done in {time.time() - t_bm25:.1f}s")

# ---- Bi-encoder embeddings (single GPU, no DataParallel) ----
# DataParallel adds ~5x overhead for SentenceTransformer encoding.
# Single GPU with large batch is much faster (~3-5s/batch vs 25s/batch).
print("Loading bi-encoder...")
model = SentenceTransformer("BAAI/bge-large-en-v1.5", device="cuda:0")
model.max_seq_length = 256  # profiles rarely exceed 256 tokens
model.half()  # fp16 — 2x faster on T4, negligible quality loss

jd_emb = model.encode([JD_QUERY], normalize_embeddings=True, batch_size=1)
np.save(os.path.join(ARTIFACTS, "jd_embedding.npy"), jd_emb[0])

print(f"Embedding {len(all_texts)} candidates (single GPU, batch={BATCH_SIZE})...")
t0 = time.time()
embeddings = model.encode(
    all_texts,
    normalize_embeddings=True,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    device="cuda:0",
)
embeddings = embeddings.astype(np.float32)
np.save(os.path.join(ARTIFACTS, "embeddings.npy"), embeddings)
print(f"Embeddings: {embeddings.shape} in {time.time() - t0:.1f}s")

# Semantic similarities
semantic_scores = embeddings @ jd_emb[0].astype(np.float32)

# ---- Cross-encoder re-ranking top-N ----
print(f"Cross-encoder re-ranking top-{CROSS_ENCODER_TOP_N}...")
del model
torch.cuda.empty_cache()

top_n_idx = np.argsort(-semantic_scores)[:CROSS_ENCODER_TOP_N]
cross_enc = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cuda:0")
ce_pairs = [(JD_QUERY, all_texts[idx]) for idx in top_n_idx]
ce_raw = cross_enc.predict(ce_pairs, batch_size=128, show_progress_bar=True)

ce_min, ce_max = float(ce_raw.min()), float(ce_raw.max())
ce_range = ce_max - ce_min if ce_max > ce_min else 1.0
cross_encoder_scores = np.zeros(len(all_texts), dtype=np.float32)
for j, idx in enumerate(top_n_idx):
    cross_encoder_scores[idx] = (float(ce_raw[j]) - ce_min) / ce_range

del cross_enc
torch.cuda.empty_cache()

# ---- Save features ----
feat_df = pd.DataFrame(feat_dicts)
feat_df.drop(columns=["profile_text"], inplace=True)
feat_df["bm25_score"] = bm25_scores
feat_df["cross_encoder_score"] = cross_encoder_scores.tolist()
feat_df.to_parquet(os.path.join(ARTIFACTS, "features.parquet"), index=False)
print(f"Features saved: {len(feat_df)} rows, {len(feat_df.columns)} cols")

In [ ]:
# Verify
emb = np.load('artifacts/embeddings.npy')
jd = np.load('artifacts/jd_embedding.npy')
df = pd.read_parquet('artifacts/features.parquet')
print(f"embeddings: {emb.shape} | jd: {jd.shape} | features: {len(df)} rows, {len(df.columns)} cols")
print(f"bm25 range: [{df.bm25_score.min():.3f}, {df.bm25_score.max():.3f}]")
print(f"cross-enc range: [{df.cross_encoder_score.min():.3f}, {df.cross_encoder_score.max():.3f}]")

# Verify LTR features exist
ltr_features = [
    "has_cs_degree", "highest_degree_level", "education_ai_relevance", "education_recency",
    "cert_count", "ml_cert_count", "cert_recency",
    "ai_title_count", "title_progression", "avg_tenure_months", "num_roles",
    "max_company_size_ord", "current_company_size_ord",
    "total_skill_count", "avg_skill_proficiency", "endorsed_skill_ratio", "skill_keyword_density",
    "work_mode_match", "search_appearance_30d", "salary_range_width", "platform_tenure_days",
]
# v2 features
v2_features = [
    "headline_has_ai_keywords", "headline_has_generic_filler",
    "salary_inverted", "salary_fits_role",
    "assessment_count", "assessment_jd_count", "assessment_proficiency_gap",
    "market_demand_score",
    "summary_is_template", "summary_ai_keyword_count",
    "career_desc_title_mismatch_count", "career_production_keyword_density",
]
all_features = ltr_features + v2_features
missing = [f for f in all_features if f not in df.columns]
assert not missing, f"Missing features: {missing}"
print(f"All {len(all_features)} LTR features present (incl. {len(v2_features)} v2)")

# Verify reasoning fields exist
reasoning_fields = ["notable_company", "best_institution", "best_field"]
missing_r = [f for f in reasoning_fields if f not in df.columns]
assert not missing_r, f"Missing reasoning fields: {missing_r}"
notable_pct = (df['notable_company'].str.len() > 0).mean() * 100
has_inst = (df['best_institution'].str.len() > 0).mean() * 100
print(f"Reasoning fields: notable_company={notable_pct:.1f}% non-empty, best_institution={has_inst:.1f}% non-empty")

# v2 feature stats
print(f"\nv2 feature stats:")
print(f"  headline_has_ai_keywords: {df['headline_has_ai_keywords'].sum()} True ({df['headline_has_ai_keywords'].mean()*100:.1f}%)")
print(f"  salary_inverted: {df['salary_inverted'].sum()} True ({df['salary_inverted'].mean()*100:.1f}%)")
print(f"  summary_is_template: {df['summary_is_template'].sum()} True ({df['summary_is_template'].mean()*100:.1f}%)")
print(f"  assessment_count: mean={df['assessment_count'].mean():.1f}")
print(f"  market_demand_score: mean={df['market_demand_score'].mean():.3f}")
print(f"  salary_fits_role: mean={df['salary_fits_role'].mean():.3f}")

# Quick stats on key features
for feat in ["has_cs_degree", "ml_cert_count", "ai_title_count", "avg_tenure_months", "work_mode_match"]:
    if df[feat].dtype == bool:
        print(f"  {feat}: {df[feat].sum()} True ({df[feat].mean()*100:.1f}%)")
    else:
        print(f"  {feat}: mean={df[feat].mean():.2f}, min={df[feat].min()}, max={df[feat].max()}")
print("Artifacts OK")

## Step 4 — Two-Level XGBoost LTR Ranking (CPU, <3 min)

**Level 1:** Hard filter on `technical_yoe >= 5` (excludes consulting years).  
**Level 2:** XGBoost LTR trains on qualified pool only, domain-focused pseudo-labels.

In [ ]:
!python src/rank.py --artifacts ./artifacts --out ./jashwanth_s.csv --method xgboost --tune --tune-trials 100

## Step 5 — Validate & Download

In [ ]:
import pandas as pd, sys
from dataclasses import fields
from collections import Counter

df = pd.read_csv('jashwanth_s.csv')
assert len(df) == 100
assert set(df['rank']) == set(range(1, 101))
scores = df.sort_values('rank')['score'].tolist()
assert all(scores[i] >= scores[i+1] for i in range(len(scores)-1))

# Tie-break check
rows = df.sort_values('rank')[['rank','score','candidate_id']].values.tolist()
for i in range(len(rows)-1):
    if rows[i][1] == rows[i+1][1]:
        assert rows[i][2] < rows[i+1][2], f"Tie-break fail at ranks {rows[i][0]},{rows[i+1][0]}"

sys.path.insert(0, '.')
from src.features import CandidateFeatures
from src.honeypot import is_honeypot
feat_df = pd.read_parquet('artifacts/features.parquet')
top_feats = feat_df[feat_df['candidate_id'].isin(set(df['candidate_id']))]

# Honeypot check (uses getattr for backward compat)
hp = sum(1 for _, r in top_feats.iterrows()
         if is_honeypot(CandidateFeatures(**{f.name: r[f.name] for f in fields(CandidateFeatures) if f.name in r.index}, profile_text='')))

# Technical YoE check
if 'technical_yoe' in top_feats.columns:
    min_tech_yoe = top_feats['technical_yoe'].min()
    below_5 = (top_feats['technical_yoe'] < 5.0).sum()
    print(f"Technical YoE: min={min_tech_yoe:.1f} | Below 5yr: {below_5}")
    assert below_5 == 0, f"{below_5} candidates with technical_yoe < 5 in top 100!"

# New feature quality checks on top 100
if 'has_cs_degree' in top_feats.columns:
    cs_pct = top_feats['has_cs_degree'].mean() * 100
    avg_ai_titles = top_feats['ai_title_count'].mean()
    avg_tenure = top_feats['avg_tenure_months'].mean()
    avg_ml_certs = top_feats['ml_cert_count'].mean()
    print(f"Top-100 profile: CS degree={cs_pct:.0f}% | Avg AI titles={avg_ai_titles:.1f} | Avg tenure={avg_tenure:.0f}mo | Avg ML certs={avg_ml_certs:.1f}")

# v2 feature checks on top 100
if 'headline_has_ai_keywords' in top_feats.columns:
    ai_headline_pct = top_feats['headline_has_ai_keywords'].mean() * 100
    template_pct = top_feats['summary_is_template'].mean() * 100
    sal_inv_pct = top_feats['salary_inverted'].mean() * 100
    avg_sal_fit = top_feats['salary_fits_role'].mean()
    avg_mkt = top_feats['market_demand_score'].mean()
    print(f"v2 signals: AI headline={ai_headline_pct:.0f}% | Template={template_pct:.0f}% | Sal inverted={sal_inv_pct:.0f}% | Sal fit={avg_sal_fit:.2f} | Mkt demand={avg_mkt:.3f}")

# Reasoning diversity check (Stage 4 anti-template)
reasons = df.sort_values('rank')['reasoning'].tolist()
lead_patterns = Counter()
for r in reasons:
    if "retrieval/ranking specialist" in r[:80]:
        lead_patterns["Retrieval specialist (v2)..."] += 1
    elif r.startswith("Former "):
        lead_patterns["Former [Company]..."] += 1
    elif " graduate with " in r:
        lead_patterns["[Institution] graduate..."] += 1
    elif "production ML deployment" in r:
        lead_patterns["Shipped/production..."] += 1
    elif "certifications" in r[:80]:
        lead_patterns["ML certs..."] += 1
    elif "ranking and retrieval" in r[:100]:
        lead_patterns["Retrieval depth..."] += 1
    elif "applied AI/ML" in r[:100]:
        lead_patterns["AI depth..."] += 1
    elif "Borderline fit:" in r[:20]:
        lead_patterns["Borderline..."] += 1
    elif "Adequate but not strong:" in r[:30]:
        lead_patterns["Adequate..."] += 1
    else:
        lead_patterns["Other"] += 1
print(f"\nReasoning lead diversity ({len(set(reasons))} unique out of 100):")
for pattern, count in lead_patterns.most_common():
    print(f"  {pattern}: {count}")
assert len(set(reasons)) == 100, "Duplicate reasoning strings found!"

print(f"\nRows: {len(df)} | Scores: {scores[0]:.4f} -> {scores[-1]:.4f} | Honeypots: {hp}")
assert hp == 0
print("All checks passed.")

In [5]:
pd.set_option('display.max_colwidth', 80)
df.sort_values('rank').head(10)[['rank', 'candidate_id', 'score', 'reasoning']]

,rank,candidate_id,score,reasoning
0,1,CAND_0011687,1.0000,Senior NLP Engineer at Niramai with 7+ years building ranking and retrieval ...
1,2,CAND_0020877,1.0000,Applied ML Engineer at CRED with 3+ years building ranking and retrieval sys...
2,3,CAND_0023458,1.0000,AI Research Engineer at BYJU'S with 5 years of applied AI/ML at product comp...
3,4,CAND_0069248,1.0000,AI Research Engineer at Unacademy with 1+ years building ranking and retriev...
4,5,CAND_0081846,1.0000,Lead AI Engineer at Razorpay with 6+ years building ranking and retrieval sy...
5,6,CAND_0022852,0.9999,AI Research Engineer at Zomato with 1+ years building ranking and retrieval ...
6,7,CAND_0030031,0.9999,AI Engineer at Microsoft with 3+ years building ranking and retrieval system...
7,8,CAND_0037160,0.9999,Data Scientist at Haptik with 1+ years building ranking and retrieval system...
8,9,CAND_0042690,0.9999,Senior Software Engineer (ML) at Niramai with 6 years of applied AI/ML at pr...
9,10,CAND_0078508,0.9999,AI Research Engineer at Rephrase.ai with 5 years of applied AI/ML at product...


In [ ]:
# XGBoost feature importance — check if new features contribute
import json
tuned = json.load(open('artifacts/tuned_params.json'))
print("Tuned params:", json.dumps(tuned, indent=2))

# Top-20 feature importance by gain (from last XGBoost run)
# Re-train quickly to get feature importances
import xgboost as xgb
from src.rank import LTR_FEATURE_COLS, create_pseudo_labels

feat_df = pd.read_parquet('artifacts/features.parquet')
jd_emb = np.load('artifacts/jd_embedding.npy')
emb = np.load('artifacts/embeddings.npy')
feat_df['semantic_sim'] = emb @ jd_emb.astype(np.float32)

# Derive has_notable_company for LTR
if 'notable_company' in feat_df.columns:
    feat_df['has_notable_company'] = (feat_df['notable_company'].fillna('').str.len() > 0).astype(int)
else:
    feat_df['has_notable_company'] = 0

from src.scoring import compute_score
from src.rank import df_row_to_features
feat_df['heuristic_score'] = [compute_score(df_row_to_features(feat_df.iloc[i]), feat_df.iloc[i].get('semantic_sim', 0)) for i in range(len(feat_df))]

available_cols = [c for c in LTR_FEATURE_COLS if c in feat_df.columns]
X = feat_df[available_cols].fillna(0).replace([np.inf, -np.inf], 0)
for col in X.columns:
    if X[col].dtype == bool:
        X[col] = X[col].astype(int)

tech_yoe = feat_df['technical_yoe'] if 'technical_yoe' in feat_df.columns else feat_df['yoe']
mask = tech_yoe >= 5.0
X_q = X[mask].values.astype(np.float32)
y = create_pseudo_labels(feat_df)
y_q = y[mask.values]

params = tuned.get('params', {})
params.pop('framework', None)
dtrain = xgb.DMatrix(X_q, label=y_q, group=[len(y_q)])
bst = xgb.train({**params, 'objective': params.get('xgb_objective', 'rank:pairwise'), 'eval_metric': 'ndcg@10'}, dtrain, num_boost_round=tuned.get('n_rounds', 20))

NEW_FEATURES = {
    # v1 features
    "has_cs_degree", "highest_degree_level", "education_ai_relevance", "education_recency",
    "cert_count", "ml_cert_count", "cert_recency",
    "ai_title_count", "title_progression", "avg_tenure_months", "num_roles",
    "max_company_size_ord", "current_company_size_ord",
    "total_skill_count", "avg_skill_proficiency", "endorsed_skill_ratio", "skill_keyword_density",
    "work_mode_match", "search_appearance_30d", "salary_range_width", "platform_tenure_days",
    "has_notable_company",
    # v2 features
    "headline_has_ai_keywords", "headline_has_generic_filler",
    "salary_inverted", "salary_fits_role",
    "assessment_count", "assessment_jd_count", "assessment_proficiency_gap",
    "market_demand_score",
    "summary_is_template", "summary_ai_keyword_count",
    "career_desc_title_mismatch_count", "career_production_keyword_density",
}

imp = bst.get_score(importance_type='gain')
sorted_imp = sorted(imp.items(), key=lambda x: -x[1])
print(f"\nTop-20 features by gain ({len(available_cols)} total):")
for fname, gain in sorted_imp[:20]:
    idx = int(fname.replace('f', ''))
    real_name = available_cols[idx]
    tag = ""
    if real_name in NEW_FEATURES:
        tag = " [v2]" if real_name in {"headline_has_ai_keywords", "headline_has_generic_filler",
            "salary_inverted", "salary_fits_role", "assessment_count", "assessment_jd_count",
            "assessment_proficiency_gap", "market_demand_score", "summary_is_template",
            "summary_ai_keyword_count", "career_desc_title_mismatch_count",
            "career_production_keyword_density"} else " [v1]"
    print(f"  {real_name}: {gain:.1f}{tag}")

In [ ]:
!python tests/validate_submission.py jashwanth_s.csv

In [ ]:
!python run_pipeline.py --run_sample sample.json --cpu --out sample_output.csv

In [ ]:
from IPython.display import FileLink
FileLink('jashwanth_s.csv')